In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

# Project root
PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT /"retail-demand-intelligence"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import (
    cleaned_data,
    master_data,
    clean_master_data
)

from src.features import (
    create_calendar_features,
    promotion_features,
    create_forecast_lag_features
)

print("Imports successful")
print("Project root:", PROJECT_ROOT)

Imports successful
Project root: c:\Users\Acer\Desktop\learning\retail-demand-intelligence


In [6]:
MODEL_PATH = PROJECT_ROOT / "models" / "catboost_final.cbm"

final_model = CatBoostRegressor()

final_model.load_model(MODEL_PATH)

print("=" * 60)
print("FINAL CATBOOST MODEL LOADED")
print("=" * 60)
print("Model:", MODEL_PATH)
print("Trees:", final_model.tree_count_)
print("Features:", len(final_model.feature_names_))

print("\nFeature names:")
print(final_model.feature_names_)

FINAL CATBOOST MODEL LOADED
Model: c:\Users\Acer\Desktop\learning\retail-demand-intelligence\models\catboost_final.cbm
Trees: 850
Features: 49

Feature names:
['store_nbr', 'family', 'onpromotion', 'dcoilwtico', 'event_count', 'has_holiday', 'has_event', 'has_additional', 'has_transfer', 'has_bridge', 'has_workday', 'is_national', 'is_regional', 'is_local', 'city', 'state', 'type', 'cluster', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21', 'sales_lag_28', 'sales_lag_35', 'sales_lag_56', 'sales_mean_7', 'sales_std_7', 'sales_mean_14', 'sales_std_14', 'sales_mean_28', 'sales_std_28', 'zero_sales_pct_28', 'sales_trend_7_28', 'has_promotion', 'promotion_days_7', 'promotion_days_28']


In [7]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"

data = {
    "train": pd.read_csv(RAW_DIR / "train.csv"),
    "test": pd.read_csv(RAW_DIR / "test.csv"),
    "oil": pd.read_csv(RAW_DIR / "oil.csv"),
    "holidays_events": pd.read_csv(RAW_DIR / "holidays_events.csv"),
    "stores": pd.read_csv(RAW_DIR / "stores.csv"),
    "transactions": pd.read_csv(RAW_DIR / "transactions.csv"),
}

for name, frame in data.items():
    print(f"{name:20s} {frame.shape}")

train                (3000888, 6)
test                 (28512, 5)
oil                  (1218, 2)
holidays_events      (350, 6)
stores               (54, 5)
transactions         (83488, 3)


In [8]:
data_clean = cleaned_data(data)

print("Cleaning complete")

for name, frame in data_clean.items():
    print(f"{name:20s} {frame.shape}")

Cleaning complete
train                (3000888, 6)
test                 (28512, 5)
oil                  (1218, 2)
holidays_events      (312, 11)
stores               (54, 5)
transactions         (83488, 3)


In [9]:
test_master = master_data(
    data_clean,
    dataset="test"
)

test_master = clean_master_data(test_master)

print("Test master:", test_master.shape)
print(test_master.head())

Test master: (28512, 21)
        id       date  store_nbr      family  onpromotion  dcoilwtico  \
0  3000888 2017-08-16          1  AUTOMOTIVE            0        46.8   
1  3000889 2017-08-16          1   BABY CARE            0        46.8   
2  3000890 2017-08-16          1      BEAUTY            2        46.8   
3  3000891 2017-08-16          1   BEVERAGES           20        46.8   
4  3000892 2017-08-16          1       BOOKS            0        46.8   

   event_count has_holiday has_event has_additional  ... has_bridge  \
0          0.0       False     False          False  ...      False   
1          0.0       False     False          False  ...      False   
2          0.0       False     False          False  ...      False   
3          0.0       False     False          False  ...      False   
4          0.0       False     False          False  ...      False   

  has_workday is_national is_regional is_local   city      state type cluster  \
0       False       False   

In [10]:
test_features = create_calendar_features(test_master)

test_features = promotion_features(test_features)

print("Test feature shape:", test_features.shape)
print("Date range:", test_features["date"].min(), "to", test_features["date"].max())

print("\nColumns:")
print(test_features.columns.tolist())

Test feature shape: (28512, 37)
Date range: 2017-08-16 00:00:00 to 2017-08-31 00:00:00

Columns:
['id', 'date', 'store_nbr', 'family', 'onpromotion', 'dcoilwtico', 'event_count', 'has_holiday', 'has_event', 'has_additional', 'has_transfer', 'has_bridge', 'has_workday', 'is_national', 'is_regional', 'is_local', 'city', 'state', 'type', 'cluster', 'transactions', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'has_promotion', 'promotion_days_7', 'promotion_days_28']


In [11]:
# Historical sales used to generate recursive lag features

history = data_clean["train"][
    ["date", "store_nbr", "family", "sales"]
].copy()

history["date"] = pd.to_datetime(history["date"])

history = history.sort_values(
    ["store_nbr", "family", "date"]
)

print("Historical sales:", history.shape)
print("Date range:", history["date"].min(), "to", history["date"].max())

Historical sales: (3000888, 4)
Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00


In [12]:
history = (
    history
    .groupby(
        ["store_nbr", "family"],
        sort=False
    )
    .tail(56)
    .reset_index(drop=True)
)

print("History after trimming:", history.shape)

History after trimming: (99792, 4)


In [13]:
first_date = test_features["date"].min()

first_day = test_features.loc[
    test_features["date"] == first_date
].copy()

print("First forecast day:", first_date)
print("Rows:", len(first_day))

first_day_features = create_forecast_lag_features(
    history,
    first_day
)

print("\nGenerated features:", first_day_features.shape)

missing_features = [
    col
    for col in final_model.feature_names_
    if col not in first_day_features.columns
]

print("Missing model features:", missing_features)

First forecast day: 2017-08-16 00:00:00
Rows: 1782

Generated features: (1782, 52)
Missing model features: []


In [14]:
X_first_day = first_day_features[
    final_model.feature_names_
]

print("Model input shape:", X_first_day.shape)
print("Expected features:", len(final_model.feature_names_))

print("\nMissing values:")
print(X_first_day.isna().sum().sort_values(ascending=False).head(10))

Model input shape: (1782, 49)
Expected features: 49

Missing values:
promotion_days_7     1782
promotion_days_28    1782
onpromotion             0
family                  0
event_count             0
has_holiday             0
has_event               0
dcoilwtico              0
has_transfer            0
has_bridge              0
dtype: int64


In [15]:
# ============================================================
# PROMOTION HISTORY FOR RECURSIVE FORECASTING
# ============================================================

promotion_history = data_clean["train"][
    ["date", "store_nbr", "family", "onpromotion"]
].copy()

promotion_history["date"] = pd.to_datetime(
    promotion_history["date"]
)

promotion_history["has_promotion"] = (
    promotion_history["onpromotion"] > 0
).astype("int8")

promotion_history = promotion_history.sort_values(
    ["store_nbr", "family", "date"]
)

# Keep only the history needed for 28-day rolling features
promotion_history = (
    promotion_history
    .groupby(
        ["store_nbr", "family"],
        sort=False
    )
    .tail(28)
    .reset_index(drop=True)
)

print("Promotion history:", promotion_history.shape)

Promotion history: (49896, 5)


In [16]:
def add_recursive_promotion_features(
    history,
    forecast_rows
):
    """
    Calculate promotion_days_7 and promotion_days_28
    using previous known promotion values.
    """

    result = forecast_rows.copy()

    # Create lookup of historical promotion indicators
    promotion_lookup = {}

    for key, group in history.groupby(
        ["store_nbr", "family"],
        sort=False
    ):
        promotion_lookup[key] = (
            group
            .sort_values("date")["has_promotion"]
            .to_numpy()
            .tolist()
        )

    promotion_7 = []
    promotion_28 = []

    for row in result.itertuples(index=False):

        key = (row.store_nbr, row.family)

        previous = promotion_lookup.get(key, [])

        # Only previous days are used
        promotion_7.append(
            sum(previous[-7:])
            if len(previous) >= 7
            else np.nan
        )

        promotion_28.append(
            sum(previous[-28:])
            if len(previous) >= 28
            else np.nan
        )

    result["promotion_days_7"] = promotion_7
    result["promotion_days_28"] = promotion_28

    return result

In [17]:
first_day_with_promo = add_recursive_promotion_features(
    promotion_history,
    first_day
)

print(
    first_day_with_promo[
        [
            "store_nbr",
            "family",
            "onpromotion",
            "has_promotion",
            "promotion_days_7",
            "promotion_days_28"
        ]
    ].head(10)
)

print("\nMissing:")
print(
    first_day_with_promo[
        ["promotion_days_7", "promotion_days_28"]
    ].isna().sum()
)

     store_nbr        family  onpromotion  has_promotion  promotion_days_7  \
0            1    AUTOMOTIVE            0              0                 0   
16           1     BABY CARE            0              0                 0   
32           1        BEAUTY            2              1                 6   
48           1     BEVERAGES           20              1                 7   
64           1         BOOKS            0              0                 0   
80           1  BREAD/BAKERY           12              1                 7   
96           1   CELEBRATION            0              0                 0   
112          1      CLEANING           25              1                 7   
128          1         DAIRY           45              1                 7   
144          1          DELI           18              1                 7   

     promotion_days_28  
0                    0  
16                   0  
32                  16  
48                  28  
64              

In [18]:
first_day_features = create_forecast_lag_features(
    history,
    first_day_with_promo
)

X_first_day = first_day_features[
    final_model.feature_names_
]

print("Model input:", X_first_day.shape)

print("\nMissing model features:")
print(
    X_first_day.isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

Model input: (1782, 49)

Missing model features:
store_nbr         0
family            0
onpromotion       0
dcoilwtico        0
event_count       0
has_holiday       0
has_event         0
has_additional    0
has_transfer      0
has_bridge        0
dtype: int64


In [19]:
print("Model input:", X_first_day.shape)
print("Expected:", (1782, 49))

missing = X_first_day.isna().sum()

print("\nMissing model features:")
print(missing[missing > 0])

print("\nTotal missing values:", missing.sum())

Model input: (1782, 49)
Expected: (1782, 49)

Missing model features:
Series([], dtype: int64)

Total missing values: 0


In [20]:
# ============================================================
# FINAL RECURSIVE FORECAST
# 2017-08-16 -> 2017-08-31
# ============================================================

TEST_START = pd.Timestamp("2017-08-16")
TEST_END = pd.Timestamp("2017-08-31")

# ------------------------------------------------------------
# Sales history
# ------------------------------------------------------------

history = data_clean["train"][
    ["date", "store_nbr", "family", "sales"]
].copy()

history["date"] = pd.to_datetime(history["date"])

history = history.sort_values(
    ["store_nbr", "family", "date"]
)

# ------------------------------------------------------------
# Promotion history
# ------------------------------------------------------------

promotion_history = data_clean["train"][
    ["date", "store_nbr", "family", "onpromotion"]
].copy()

promotion_history["date"] = pd.to_datetime(
    promotion_history["date"]
)

promotion_history["has_promotion"] = (
    promotion_history["onpromotion"] > 0
).astype("int8")

promotion_history = promotion_history.sort_values(
    ["store_nbr", "family", "date"]
)

# ------------------------------------------------------------
# Test data
# ------------------------------------------------------------

forecast = test_features[
    (test_features["date"] >= TEST_START) &
    (test_features["date"] <= TEST_END)
].copy()

forecast = forecast.sort_values(
    ["date", "store_nbr", "family"]
).reset_index(drop=True)

print("=" * 60)
print("FINAL RECURSIVE FORECAST")
print("=" * 60)
print("Rows:", len(forecast))
print("Dates:", forecast["date"].min(), "->", forecast["date"].max())

predictions = []

# ------------------------------------------------------------
# Recursive daily forecasting
# ------------------------------------------------------------

for current_date in forecast["date"].drop_duplicates():

    print(f"Processing {current_date.date()}...")

    # Current day's rows
    forecast_rows = forecast.loc[
        forecast["date"] == current_date
    ].copy()

    # --------------------------------------------
    # Sales lag / rolling features
    # --------------------------------------------

    forecast_features = create_forecast_lag_features(
        history,
        forecast_rows
    )

    # --------------------------------------------
    # Promotion rolling features
    # --------------------------------------------

    forecast_features = add_recursive_promotion_features(
        promotion_history,
        forecast_features
    )

    # --------------------------------------------
    # Exact model feature order
    # --------------------------------------------

    X_current = forecast_features[
        final_model.feature_names_
    ]

    # Safety check
    if X_current.isna().any().any():
        missing_counts = X_current.isna().sum()
        print(
            "WARNING — missing features:"
        )
        print(
            missing_counts[
                missing_counts > 0
            ]
        )
        raise ValueError(
            f"Missing model features on {current_date.date()}"
        )

    # --------------------------------------------
    # Predict
    # --------------------------------------------

    day_predictions = final_model.predict(
        X_current
    )

    # Sales cannot be negative
    day_predictions = np.maximum(
        day_predictions,
        0
    )

    # --------------------------------------------
    # Store predictions
    # --------------------------------------------

    day_result = forecast_rows[
        ["id", "date", "store_nbr", "family"]
    ].copy()

    day_result["prediction"] = day_predictions

    predictions.append(day_result)

    # --------------------------------------------
    # Update SALES history
    # --------------------------------------------

    history_update = forecast_rows[
        ["date", "store_nbr", "family"]
    ].copy()

    history_update["sales"] = day_predictions

    history = pd.concat(
        [history, history_update],
        ignore_index=True
    )

    history = (
        history
        .sort_values(
            ["store_nbr", "family", "date"]
        )
        .groupby(
            ["store_nbr", "family"],
            sort=False
        )
        .tail(56)
        .reset_index(drop=True)
    )

    # --------------------------------------------
    # Update PROMOTION history
    # --------------------------------------------

    promotion_update = forecast_rows[
        [
            "date",
            "store_nbr",
            "family",
            "has_promotion"
        ]
    ].copy()

    promotion_history = pd.concat(
        [
            promotion_history,
            promotion_update
        ],
        ignore_index=True
    )

    promotion_history = (
        promotion_history
        .sort_values(
            ["store_nbr", "family", "date"]
        )
        .groupby(
            ["store_nbr", "family"],
            sort=False
        )
        .tail(28)
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# Combine predictions
# ------------------------------------------------------------

final_predictions = pd.concat(
    predictions,
    ignore_index=True
)

print("\n" + "=" * 60)
print("FORECAST COMPLETE")
print("=" * 60)

print("Rows:", len(final_predictions))
print(
    "Date range:",
    final_predictions["date"].min(),
    "->",
    final_predictions["date"].max()
)

print("\nPrediction statistics:")
print(
    final_predictions["prediction"].describe()
)

FINAL RECURSIVE FORECAST
Rows: 28512
Dates: 2017-08-16 00:00:00 -> 2017-08-31 00:00:00
Processing 2017-08-16...
Processing 2017-08-17...
Processing 2017-08-18...
Processing 2017-08-19...
Processing 2017-08-20...
Processing 2017-08-21...
Processing 2017-08-22...
Processing 2017-08-23...
Processing 2017-08-24...
Processing 2017-08-25...
Processing 2017-08-26...
Processing 2017-08-27...
Processing 2017-08-28...
Processing 2017-08-29...
Processing 2017-08-30...
Processing 2017-08-31...

FORECAST COMPLETE
Rows: 28512
Date range: 2017-08-16 00:00:00 -> 2017-08-31 00:00:00

Prediction statistics:
count    28512.000000
mean       440.817436
std       1143.277604
min          0.000000
25%          9.112542
50%         36.288140
75%        278.618045
max      13225.851025
Name: prediction, dtype: float64


In [21]:
print(
    final_predictions
    .groupby("date")["prediction"]
    .agg(
        mean="mean",
        median="median",
        min="min",
        max="max",
        total="sum"
    )
)

                  mean     median       min           max          total
date                                                                    
2017-08-16  468.945957  49.622263  0.000000  11801.778854  835661.695450
2017-08-17  374.765855  35.088822  1.021712   7747.183774  667832.752724
2017-08-18  435.471876  42.264545  1.254680   9884.425756  776010.883114
2017-08-19  512.387801  47.559104  0.000000  11844.586982  913075.061304
2017-08-20  549.619054  41.714960  0.000000  13225.851025  979421.155060
2017-08-21  428.822048  34.311292  2.082754   8969.788621  764160.889647
2017-08-22  405.873796  34.249414  4.726534   7885.516873  723267.103774
2017-08-23  426.952448  37.081338  1.670569  12390.032384  760829.263030
2017-08-24  357.000898  29.923754  2.121175   7497.707339  636175.600915
2017-08-25  419.141233  33.890958  0.000000   9872.692665  746909.677467
2017-08-26  500.813876  36.287552  0.000000  11176.366240  892450.327634
2017-08-27  532.728879  33.277853  0.000000  12301.

In [23]:
print("\nFirst 20 predictions:")
print(final_predictions.head(20))

print("\nLast 20 predictions:")
print(final_predictions.tail(20))


First 20 predictions:
         id       date  store_nbr               family   prediction
0   3000888 2017-08-16          1           AUTOMOTIVE    15.672622
1   3000889 2017-08-16          1            BABY CARE     9.664695
2   3000890 2017-08-16          1               BEAUTY    19.027953
3   3000891 2017-08-16          1            BEVERAGES  2184.442018
4   3000892 2017-08-16          1                BOOKS     9.618840
5   3000893 2017-08-16          1         BREAD/BAKERY   385.884977
6   3000894 2017-08-16          1          CELEBRATION    19.610719
7   3000895 2017-08-16          1             CLEANING   684.768034
8   3000896 2017-08-16          1                DAIRY   744.311256
9   3000897 2017-08-16          1                 DELI   148.029811
10  3000898 2017-08-16          1                 EGGS   157.545494
11  3000899 2017-08-16          1         FROZEN FOODS   154.861978
12  3000900 2017-08-16          1            GROCERY I  2594.848413
13  3000901 2017-08-16   

In [24]:
print("Rows:", len(final_predictions))
print("Unique IDs:", final_predictions["id"].nunique())
print("Missing predictions:", final_predictions["prediction"].isna().sum())
print("Negative predictions:", (final_predictions["prediction"] < 0).sum())

print(
    "Date range:",
    final_predictions["date"].min(),
    "->",
    final_predictions["date"].max()
)

Rows: 28512
Unique IDs: 28512
Missing predictions: 0
Negative predictions: 0
Date range: 2017-08-16 00:00:00 -> 2017-08-31 00:00:00


In [25]:
daily_summary = (
    final_predictions
    .groupby("date")["prediction"]
    .agg(
        total="sum",
        mean="mean",
        median="median",
        max="max"
    )
)

display(daily_summary)

,total,mean,median,max
date,,,,
2017-08-16,835661.695450,468.945957,49.622263,11801.778854
2017-08-17,667832.752724,374.765855,35.088822,7747.183774
2017-08-18,776010.883114,435.471876,42.264545,9884.425756
2017-08-19,913075.061304,512.387801,47.559104,11844.586982
2017-08-20,979421.155060,549.619054,41.714960,13225.851025
2017-08-21,764160.889647,428.822048,34.311292,8969.788621
2017-08-22,723267.103774,405.873796,34.249414,7885.516873
2017-08-23,760829.263030,426.952448,37.081338,12390.032384
2017-08-24,636175.600915,357.000898,29.923754,7497.707339


In [26]:
daily_summary = (
    final_predictions
    .groupby("date")["prediction"]
    .agg(
        total="sum",
        mean="mean",
        median="median",
        max="max"
    )
)

print(daily_summary)

                    total        mean     median           max
date                                                          
2017-08-16  835661.695450  468.945957  49.622263  11801.778854
2017-08-17  667832.752724  374.765855  35.088822   7747.183774
2017-08-18  776010.883114  435.471876  42.264545   9884.425756
2017-08-19  913075.061304  512.387801  47.559104  11844.586982
2017-08-20  979421.155060  549.619054  41.714960  13225.851025
2017-08-21  764160.889647  428.822048  34.311292   8969.788621
2017-08-22  723267.103774  405.873796  34.249414   7885.516873
2017-08-23  760829.263030  426.952448  37.081338  12390.032384
2017-08-24  636175.600915  357.000898  29.923754   7497.707339
2017-08-25  746909.677467  419.141233  33.890958   9872.692665
2017-08-26  892450.327634  500.813876  36.287552  11176.366240
2017-08-27  949322.862235  532.728879  33.277853  12301.058827
2017-08-28  733047.919008  411.362469  28.306999   8361.692007
2017-08-29  714863.923938  401.158206  29.165481   7790

In [30]:
submission = (
    final_predictions[["id", "prediction"]]
    .rename(columns={"prediction": "unit_sales"})
    .sort_values("id")
    .reset_index(drop=True)
)

print(submission.head())
print(submission.tail())
print("\nShape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Missing:", submission["unit_sales"].isna().sum())

        id   unit_sales
0  3000888    15.672622
1  3000889     9.664695
2  3000890    19.027953
3  3000891  2184.442018
4  3000892     9.618840
            id   unit_sales
28507  3029395   314.079226
28508  3029396    96.099494
28509  3029397  1185.150651
28510  3029398    81.794689
28511  3029399    10.218025

Shape: (28512, 2)
Columns: ['id', 'unit_sales']
Missing: 0


In [31]:
submission_path = PROJECT_ROOT / "outputs" / "submission_catboost_final.csv"

submission_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

submission.to_csv(
    submission_path,
    index=False
)

print("\nSaved:")
print(submission_path)


Saved:
c:\Users\Acer\Desktop\learning\retail-demand-intelligence\outputs\submission_catboost_final.csv
